# Held-out evaluation
Run once after full training and validation-based model selection are complete. No test metrics exist until this notebook actually evaluates an approved non-smoke model.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
from ml.evaluation.evaluate import evaluate
MODEL_SELECTION_LOCKED = False
if MODEL_SELECTION_LOCKED:
    metrics = evaluate("artifacts/checkpoints/run/best.pt", config, selection_locked=True)
    display(metrics)
else:
    print("Test evaluation is locked until model selection is final.")

## Error analysis
Review false positives and false negatives only after final evaluation. These errors describe this dataset, not clinical reliability. Class imbalance, pediatric acquisition patterns, uncertain patient identity, and hospital/domain shift limit generalization.

In [ ]:
import pandas as pd
path = Path("artifacts/reports/predictions.csv")
if path.exists():
    predictions = pd.read_csv(path)
    errors = predictions[predictions.target != predictions.prediction]
    display(errors)
    display(errors.groupby(["target", "prediction"]).size())